# V3: Qwen3.5-4B LoRA on the verified v3 data (12,399 rows)

Same recipe as V2 (LoRA r32/alpha64 on all linear layers, assistant-only loss, weight-0 turns masked, effective batch 16, 1 epoch, lr 2e-4 cosine).

1. Upload `data/v3/train.jsonl` to `MyDrive/data-sft/v3/train.jsonl`.
2. Runtime: **A100**.
3. Run cells top to bottom **up to and including the cost check**, and stop there.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate wandb

In [ ]:
!pip install -q flash-linear-attention

In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install causal_conv1d

In [ ]:
!pip install -q liger-kernel

In [ ]:
MODEL      = "Qwen/Qwen3.5-4B"
DTYPE      = "bf16"          # A100
ATTN       = "sdpa"
MAX_LEN    = 8192
N_ROWS     = None            # ALL rows
NUM_EPOCHS = 1
OUT_DIR    = "/content/drive/MyDrive/data-sft/runs/qwen4b-v3"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
from datasets import Dataset

DATA_DIR = "/content/drive/MyDrive/data-sft/v3"   # upload data/v3/train.jsonl here

def load_rows(path):
    """Keep only what training needs. Tool-call arguments become JSON strings
    (see note 1 below for why)."""
    rows = []
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            msgs = []
            for m in r["messages"]:
                m = dict(m)
                if m.get("tool_calls"):
                    m["tool_calls"] = [
                        {**tc, "function": {**tc["function"],
                                            "arguments": json.dumps(tc["function"]["arguments"], ensure_ascii=False)}}
                        for tc in m["tool_calls"]
                    ]
                msgs.append(m)
            rows.append({"messages": msgs, "tools": r["tools"]})
    return rows

rows = load_rows(f"{DATA_DIR}/train.jsonl")
if N_ROWS:                                   # from your config cell (None = all)
    rows = rows[:N_ROWS]

ds = Dataset.from_list(rows)
split = ds.train_test_split(test_size=0.02, seed=42)   # ~2% for validation loss
train_ds, val_ds = split["train"], split["test"]

print(train_ds, val_ds)
print(json.dumps(train_ds[0]["messages"], indent=1, ensure_ascii=False)[:2000])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
print(type(tokenizer).__name__, "| vocab:", len(tokenizer))

In [ ]:
def for_template(messages):
    """Undo storage tricks before rendering: arguments back to dicts,
    and drop the None fields Arrow added (tool_calls=None, weight=None, ...)."""
    out = []
    for m in messages:
        m = {k: v for k, v in m.items() if v is not None}        # remove Arrow's None padding
        if m.get("tool_calls"):
            m["tool_calls"] = [
                {**tc, "function": {**tc["function"],
                                    "arguments": json.loads(tc["function"]["arguments"])}}
                for tc in m["tool_calls"]
            ]
        out.append(m)
    return out

row = train_ds[0]
text = tokenizer.apply_chat_template(
    for_template(row["messages"]),
    tools=row["tools"],
    tokenize=False,
    enable_thinking=False,
)
print(text)

In [ ]:
import re

# One assistant turn: the header, the empty think block, then the part we train on (up to and incl. <|im_end|>)
ASSISTANT_TURN = re.compile(
    r"<\|im_start\|>assistant\n(?:<think>\n\n</think>\n\n)?(.*?<\|im_end\|>)", re.DOTALL)

def tokenize_row(row):
    msgs = for_template(row["messages"])
    text = tokenizer.apply_chat_template(msgs, tools=row["tools"], tokenize=False, enable_thinking=False)

    # character spans of each assistant turn's trainable part, in order
    spans = [m.span(1) for m in ASSISTANT_TURN.finditer(text)]
    weights = [m.get("weight", 1) for m in msgs if m["role"] == "assistant"]
    assert len(spans) == len(weights), f"found {len(spans)} assistant turns, expected {len(weights)}"
    spans = [s for s, w in zip(spans, weights) if w != 0]          # drop weight-0 turns

    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    labels = []
    for tok_id, (a, b) in zip(enc["input_ids"], enc["offset_mapping"]):
        inside = any(start <= a and b <= end for start, end in spans)
        labels.append(tok_id if inside else -100)

    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def show_mask(row):
    out = tokenize_row(row)
    pieces, inside = [], False
    for tok_id, lab in zip(out["input_ids"], out["labels"]):
        t = tokenizer.decode([tok_id])
        if lab != -100 and not inside: pieces.append("[["); inside = True
        if lab == -100 and inside:     pieces.append("]]"); inside = False
        pieces.append(t)
    if inside: pieces.append("]]")
    print("".join(pieces))
    n = sum(l != -100 for l in out["labels"])
    print(f"\n--- {len(out['input_ids'])} tokens, {n} trained ({n/len(out['input_ids']):.0%})")

show_mask(train_ds[0])

In [ ]:
idx = next(i for i in range(len(train_ds)) if any(m.get("weight") == 0 for m in train_ds[i]["messages"]))
show_mask(train_ds[idx])

In [ ]:
cols = train_ds.column_names                      # ["messages", "tools"]: replaced by token columns
train_tok = train_ds.map(tokenize_row, remove_columns=cols, num_proc=2)
val_tok   = val_ds.map(tokenize_row,   remove_columns=cols, num_proc=2)

before = len(train_tok)
train_tok = train_tok.filter(lambda r: len(r["input_ids"]) <= MAX_LEN)
val_tok   = val_tok.filter(lambda r: len(r["input_ids"]) <= MAX_LEN)
print(f"train {len(train_tok)} (dropped {before - len(train_tok)} over {MAX_LEN} tokens), val {len(val_tok)}")

lens = sorted(len(x) for x in train_tok["input_ids"])
print("tokens: median", lens[len(lens)//2], "| p99", lens[int(len(lens)*.99)], "| max", lens[-1])

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

from liger_kernel.transformers.monkey_patch import MODEL_TYPE_TO_APPLY_LIGER_FN
MODEL_TYPE_TO_APPLY_LIGER_FN["qwen3_5_text"]()

dtype = torch.bfloat16 if DTYPE == "bf16" else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype, attn_implementation=ATTN, device_map="cuda")

model.gradient_checkpointing_enable()   # trade compute for memory (see below)
model.enable_input_require_grads()      # needed when gradient checkpointing + LoRA are combined
model.config.use_cache = False          # the generation cache is useless during training

lora = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules="all-linear",        # LoRA on every linear layer (attention + MLP)
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()      # expect roughly 1-2% trainable

## Cost check (run first, then report the printed hours before starting the full run)


In [ ]:
# COST CHECK: run this before the full training cell. ~10 minutes.
# Trains 15 optimizer steps on a throwaway copy of the settings, then prints the projected time for the full run.
import time, copy
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
probe_args = TrainingArguments(output_dir="/content/probe", per_device_train_batch_size=2, gradient_accumulation_steps=8,
                               learning_rate=2e-4, max_steps=15, logging_steps=5, bf16=True, report_to="none",
                               train_sampling_strategy="group_by_length", save_strategy="no")
probe = Trainer(model=model, args=probe_args, train_dataset=train_tok.shuffle(seed=1).select(range(15 * 16)),
                data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
t0 = time.time(); probe.train(); dt = (time.time() - t0) / 15
steps = len(train_tok) // 16
print(f"{dt:.1f} s/step x {steps} steps = {dt * steps / 3600:.1f} h  (+ ~10% for evals/saves)")
print("peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB")
# The probe touched the LoRA weights: re-run the model cell (LoRA setup) before the full training cell.


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

NUM_EPOCHS = 1                                                   # ← NEW
steps_total = len(train_tok) // 16 * NUM_EPOCHS                   # ← NEW: optimizer steps (batch 16)
print("total optimizer steps:", steps_total)

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=2,           # same as V2: 2 x 8 = effective batch 16
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=int(0.03 * steps_total),     # ← CHANGED: 3% of the real total
    num_train_epochs=NUM_EPOCHS,              # ← CHANGED: replaces max_steps
    logging_steps=10,                         # ← CHANGED: 5 → 10
    eval_strategy="steps", eval_steps=100,    # ← CHANGED: 25 → 100
    save_strategy="steps", save_steps=100,    # ← CHANGED: 25 → 100
    save_total_limit=2,
    load_best_model_at_end=True,              # ← NEW: keep the checkpoint with the lowest eval_loss
    metric_for_best_model="eval_loss",        # ← NEW
    bf16=(DTYPE == "bf16"), fp16=(DTYPE == "fp16"),
    train_sampling_strategy="group_by_length",
    report_to="none",
    per_device_eval_batch_size=1,
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
trainer.train()                               # ← CHANGED: fresh start, NOT resume

In [ ]:
!ls {OUT_DIR}

In [ ]:
trainer.save_model(f"{OUT_DIR}/final")
tokenizer.save_pretrained(f"{OUT_DIR}/final")

In [ ]:
model.eval()
test = for_template(val_ds[0]["messages"])[:2]          # system + user only
prompt = tokenizer.apply_chat_template(test, tools=val_ds[0]["tools"], tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)

ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")   # text → token ids on GPU
out = model.generate(**ids, max_new_tokens=300, do_sample=False)                     # model writes up to 300 tokens

print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False))  # only the new part

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!huggingface-cli download Qwen/Qwen3.5-4B --local-dir base
!python llama.cpp/convert_hf_to_gguf.py base --outtype f16 --outfile qwen35-4b-f16.gguf
!python llama.cpp/convert_lora_to_gguf.py {OUT_DIR}/final --base base --outtype f16 --outfile lora-4b.gguf

In [ ]:
!ls -la {OUT_DIR}/final